# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [ ]:
# Verificar que Colab asignó una GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga de modelos.

In [ ]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")


In [ ]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [ ]:
# Descargar los dos modelos Llama (la primera vez tarda unos minutos)

MODELO_LIGERO = "llama3.2:3b"
MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}
!ollama pull {MODELO_GRANDE}
!ollama list


### **CLIENTE DE PYTHON**

In [ ]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")


## **DE PALABRAS A TOKENS**

### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)

print(prompt)
print(response.message.content)


### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

tokens_prompt = response.prompt_eval_count
tokens_respuesta = response.eval_count

print("Tokens del prompt:", tokens_prompt)
print("Tokens de la respuesta:", tokens_respuesta)
print("Tokens totales:", tokens_prompt + tokens_respuesta)
# Con Ollama no se factura: los tokens miden cuánto trabajo hizo el modelo y cuánto de su contexto usaste


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

inicio = time.time()
response_tiempo = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad

inicio = time.time()
response_grande = client.chat(
    model=MODELO_GRANDE,
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio
carga_grande = response_grande.load_duration / 1e9  # nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count
tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s — {tokens_ligero} tokens")
print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s — {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")
print("\nRespuesta del modelo grande:\n", response_grande.message.content)


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [ ]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")


In [ ]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version


In [ ]:
# Descargar el modelo ligero de Llama e inicializar el cliente

MODELO_LIGERO = "llama3.2:3b"

!ollama pull {MODELO_LIGERO}
!ollama list

!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado con el modelo ligero:", MODELO_LIGERO)


**2. Consulta la primera pregunta:**

* Construye una lista llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).
*  Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [ ]:
# Definir la lista de preguntas

preguntas = [
    "¿Qué es la OD680 y para qué se usa en el cultivo de microalgas?",
    "¿Cómo se calcula la tasa específica de crecimiento de un cultivo de Chlorella vulgaris?",
    "¿Qué diferencia hay entre cultivar microalgas con luz azul y con luz roja?",
]

print("Preguntas del comparador:")
for i, pregunta in enumerate(preguntas, start=1):
    print(f"{i}. {pregunta}")


In [ ]:
# Consultar la pregunta 1 y guardar el resultado en resultado_1

inicio = time.time()
respuesta_1 = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": preguntas[0]}]
)
tiempo_1 = time.time() - inicio

resultado_1 = {
    "pregunta": preguntas[0],
    "modelo": MODELO_LIGERO,
    "tiempo_segundos": round(tiempo_1, 2),
    "tokens_prompt": respuesta_1.prompt_eval_count,
    "tokens_respuesta": respuesta_1.eval_count,
    "tokens_totales": respuesta_1.prompt_eval_count + respuesta_1.eval_count,
    "respuesta": respuesta_1.message.content,
}

print("Pregunta 1:", resultado_1["pregunta"])
print(f"Tiempo: {resultado_1['tiempo_segundos']} s — Tokens: {resultado_1['tokens_totales']}")
print("\nRespuesta:\n", resultado_1["respuesta"])


**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la pregunta 2 y guardar el resultado en resultado_2

inicio = time.time()
respuesta_2 = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": preguntas[1]}]
)
tiempo_2 = time.time() - inicio

resultado_2 = {
    "pregunta": preguntas[1],
    "modelo": MODELO_LIGERO,
    "tiempo_segundos": round(tiempo_2, 2),
    "tokens_prompt": respuesta_2.prompt_eval_count,
    "tokens_respuesta": respuesta_2.eval_count,
    "tokens_totales": respuesta_2.prompt_eval_count + respuesta_2.eval_count,
    "respuesta": respuesta_2.message.content,
}

print("Pregunta 2:", resultado_2["pregunta"])
print(f"Tiempo: {resultado_2['tiempo_segundos']} s — Tokens: {resultado_2['tokens_totales']}")
print("\nRespuesta:\n", resultado_2["respuesta"])


In [ ]:
# Consultar la pregunta 3 y guardar el resultado en resultado_3

inicio = time.time()
respuesta_3 = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": preguntas[2]}]
)
tiempo_3 = time.time() - inicio

resultado_3 = {
    "pregunta": preguntas[2],
    "modelo": MODELO_LIGERO,
    "tiempo_segundos": round(tiempo_3, 2),
    "tokens_prompt": respuesta_3.prompt_eval_count,
    "tokens_respuesta": respuesta_3.eval_count,
    "tokens_totales": respuesta_3.prompt_eval_count + respuesta_3.eval_count,
    "respuesta": respuesta_3.message.content,
}

print("Pregunta 3:", resultado_3["pregunta"])
print(f"Tiempo: {resultado_3['tiempo_segundos']} s — Tokens: {resultado_3['tokens_totales']}")
print("\nRespuesta:\n", resultado_3["respuesta"])


**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios

resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

print("Resultados guardados:", len(resultados))


**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados

for resultado in resultados:
    print("=" * 70)
    print("Pregunta:", resultado["pregunta"])
    print("Modelo:", resultado["modelo"])
    print(f"Tiempo: {resultado['tiempo_segundos']} segundos")
    print("Tokens del prompt:", resultado["tokens_prompt"])
    print("Tokens de respuesta:", resultado["tokens_respuesta"])
    print("Tokens totales:", resultado["tokens_totales"])
    print("\nRespuesta:")
    print(resultado["respuesta"])
    print()

tiempos = [r["tiempo_segundos"] for r in resultados]
tokens = [r["tokens_totales"] for r in resultados]
print("=" * 70)
print(f"Tiempo promedio por pregunta: {sum(tiempos) / len(tiempos):.2f} segundos")
print(f"Tokens totales de las 3 preguntas: {sum(tokens)}")
print(f"Preguntas con respuesta generada: {sum(1 for r in resultados if r['respuesta'].strip())} de {len(resultados)}")
# La valoración cualitativa se lee de las tres respuestas anteriores: si son correctas
# y completas, el modelo ligero es suficiente para preguntas frecuentes de este tipo.
